In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ECommerce-Fraud-Detection")
    .master("local[*]")
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()
)
# Ẩn các thông báo log rác, chỉ hiển thị thông báo lỗi hệ thống
spark.sparkContext.setLogLevel("ERROR")

print("Khởi tạo SparkSession thành công")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/31 12:47:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Khởi tạo SparkSession thành công


In [2]:
# Đường dẫn lưu trữ tập dữ liệu trên hệ thống tệp tin phân tán HDFS
hdfs_path = "hdfs://localhost:9090/ecommerce-fraud-detection/transactions.csv"
# Đọc dữ liệu từ HDFS vào Spark DataFrame
df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True
)
dataframe = df.limit(5).toPandas()
display(dataframe.style.hide(axis="index"))

transaction_id,user_id,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,transaction_time,shipping_distance_km,is_fraud
1,1,141,47,147.930000,84.750000,FR,FR,web,travel,0,1,1,1,2024-01-06 11:09:39,370.950000,0
2,1,141,47,147.930000,107.900000,FR,FR,web,travel,0,0,0,0,2024-01-10 03:13:47,149.620000,0
3,1,141,47,147.930000,92.360000,FR,FR,app,travel,1,1,1,1,2024-01-12 13:20:11,164.080000,0
4,1,141,47,147.930000,112.470000,FR,FR,web,fashion,0,1,1,1,2024-01-16 00:00:04,397.400000,0
5,1,141,47,147.930000,132.910000,FR,US,web,electronics,0,1,1,1,2024-01-17 08:27:31,935.280000,0


In [3]:
df.createOrReplaceTempView("ecommerce_transactions")
spark.catalog.listTables()

[Table(name='ecommerce_transactions', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [4]:
# 1. Kiểm tra số lượng dòng ban đầu trước khi xử lý NULL
total_before = df.count()
print(f"Tổng số dòng trước khi lọc Null: {total_before:,} dòng\n")

# 2. Loại bỏ các dòng (Null) ở các cột định danh bắt buộc và cột mục tiêu
df = df.dropna(subset=["transaction_id", "user_id", "amount", "is_fraud"])

# 3. Điền giá trị mặc định cho các trường phân loại bị trống
df = df.fillna({
    "country": "Unknown",
    "bin_country": "Unknown",
    "channel": "Unknown",
    "merchant_category": "Unknown"
})
# 4. Ghi nhận số lượng dòng sau khi xử lý khuyết thiếu
total_after = df.count()
print(f"Tổng số dòng sau khi lọc Null: {total_after:,} dòng")

print(f"Số lượng dòng đã bị xóa: {total_before - total_after:,} dòng\n")

Tổng số dòng trước khi lọc Null: 299,695 dòng



Tổng số dòng sau khi lọc Null: 299,695 dòng
Số lượng dòng đã bị xóa: 0 dòng



In [5]:
# 1. Ghi nhận số lượng dòng trước khi loại bỏ trùng lặp
total_before_dup = df.count()
print(f"Tổng số dòng trước khi lọc trùng: {total_before_dup:,}")

# 2. Lọc trùng dựa trên tổ hợp hành vi giao dịch bị lặp lại trong cùng một thời điểm
df = df.dropDuplicates(subset=["user_id", "amount", "merchant_category", "transaction_time"])

# 3. Ghi nhận số lượng dòng sau khi xử lý trùng lặp
total_after_dup = df.count()
print(f"Tổng số dòng sau khi lọc trùng: {total_after_dup:,}")
print(f"Số lượng bản ghi trùng lặp bị xóa bỏ: {total_before_dup - total_after_dup}")

Tổng số dòng trước khi lọc trùng: 299,695


Tổng số dòng sau khi lọc trùng: 299,695
Số lượng bản ghi trùng lặp bị xóa bỏ: 0


In [6]:
from pyspark.sql.functions import col
# 1. Ghi nhận số lượng dòng trước khi tiến hành lọc nhiễu logic
total_before_filter = df.count()
print(f"Tổng số dòng trước khi lọc nhiễu: {total_before_filter:,}")
# 2. Áp dụng bộ lọc đa điều kiện
df = df.filter(
    (col("amount") > 0) &
    (col("avg_amount_user") > 0) &
    (col("account_age_days") >= 0) &
    (col("total_transactions_user") >= 0) &
    (col("shipping_distance_km") >= 0) &
    (col("shipping_distance_km") <= 10000) &
    (col("amount") <= (col("avg_amount_user") * 100))
)
# 3. Ghi nhận số lượng dòng sau khi thanh lọc dữ liệu bất hợp lý
total_after_filter = df.count()
print(f"Tổng số dòng sau khi lọc nhiễu: {total_after_filter:,}")
print(f"Số lượng bản ghi nhiễu logic bị loại bỏ: {total_before_filter - total_after_filter}")

Tổng số dòng trước khi lọc nhiễu: 299,695


Tổng số dòng sau khi lọc nhiễu: 299,579
Số lượng bản ghi nhiễu logic bị loại bỏ: 116


In [7]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType, TimestampType, BooleanType

# 1. Hiển thị lược đồ kiểu dữ liệu thô trước khi chuyển đổi
print("Lược đồ kiểu dữ liệu trước khi chuẩn hóa")
df.printSchema()

# 2. Thực hiện ép kiểu dữ liệu chuẩn hóa toàn diện
df_casted = df \
    .withColumn("amount", col("amount").cast(DoubleType())) \
    .withColumn("avg_amount_user", col("avg_amount_user").cast(DoubleType())) \
    .withColumn("shipping_distance_km", col("shipping_distance_km").cast(DoubleType())) \
    .withColumn("account_age_days", col("account_age_days").cast(IntegerType())) \
    .withColumn("total_transactions_user", col("total_transactions_user").cast(IntegerType())) \
    .withColumn("is_fraud", col("is_fraud").cast(IntegerType())) \
    .withColumn("transaction_time", col("transaction_time").cast(TimestampType())) \
    .withColumn("promo_used", col("promo_used").cast(BooleanType())) \
    .withColumn("avs_match", col("avs_match").cast(BooleanType())) \
    .withColumn("cvv_result", col("cvv_result").cast(BooleanType())) \
    .withColumn("three_ds_flag", col("three_ds_flag").cast(BooleanType()))

# 3. Hiển thị lược đồ sau khi đã chuẩn hóa để đối chiếu
print("Lược đồ kiểu dữ liệu sau khi chuẩn hóa")
df_casted.printSchema()

# Cập nhật lại biến df chính để truyền dữ liệu sạch sang các bước sau
df = df_casted

Lược đồ kiểu dữ liệu trước khi chuẩn hóa
root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- account_age_days: integer (nullable = true)
 |-- total_transactions_user: integer (nullable = true)
 |-- avg_amount_user: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- country: string (nullable = false)
 |-- bin_country: string (nullable = false)
 |-- channel: string (nullable = false)
 |-- merchant_category: string (nullable = false)
 |-- promo_used: integer (nullable = true)
 |-- avs_match: integer (nullable = true)
 |-- cvv_result: integer (nullable = true)
 |-- three_ds_flag: integer (nullable = true)
 |-- transaction_time: timestamp (nullable = true)
 |-- shipping_distance_km: double (nullable = true)
 |-- is_fraud: integer (nullable = true)

Lược đồ kiểu dữ liệu sau khi chuẩn hóa
root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- account_age_days: integer (nullable = true)
 |--

In [8]:
# 1. Tỷ lệ và tổng giá trị của giao dịch gian lận so với giao dịch hợp lệ là bao nhiêu?
query_1 = spark.sql("""
    SELECT
        CASE
            WHEN is_fraud = 1 THEN 'Gian lận (Fraudulent)'
            ELSE 'Hợp lệ (Legitimate)'
        END AS loai_giao_dich,
        COUNT(*) AS so_luong_giao_dich,
        CAST(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() AS DECIMAL(5, 2)) AS ty_le_phan_tram,
        SUM(amount) AS tong_gia_tri,
        AVG(amount) AS gia_tri_trung_binh
    FROM ecommerce_transactions
    GROUP BY is_fraud;
""")
query_1.show()

+--------------------+------------------+---------------+-------------------+------------------+
|      loai_giao_dich|so_luong_giao_dich|ty_le_phan_tram|       tong_gia_tri|gia_tri_trung_binh|
+--------------------+------------------+---------------+-------------------+------------------+
|Gian lận (Fraudul...|              6612|           2.21| 3907435.4500000007| 590.9611993345434|
| Hợp lệ (Legitimate)|            293083|          97.79|4.918811271000002E7|167.82997550182037|
+--------------------+------------------+---------------+-------------------+------------------+



In [9]:
# 2. Danh mục sản phẩm bị gian lận nhiều nhất về cả số lượng và tổng giá trị thất thoát là gì?
query_2 = spark.sql("""
    SELECT
        merchant_category,
        COUNT(*) AS so_luong_gian_lan,
        SUM(amount) AS tong_thiet_hai
    FROM ecommerce_transactions
    WHERE is_fraud = 1
    GROUP BY merchant_category
    ORDER BY tong_thiet_hai DESC
""")
query_2.show()

+-----------------+-----------------+-----------------+
|merchant_category|so_luong_gian_lan|   tong_thiet_hai|
+-----------------+-----------------+-----------------+
|      electronics|             1357|        840480.36|
|           travel|             1388|786640.9999999999|
|          fashion|             1340|775658.0299999998|
|           gaming|             1295|        761831.01|
|          grocery|             1232|        742825.05|
+-----------------+-----------------+-----------------+



In [10]:
# 3. Tỷ lệ gian lận thay đổi như thế nào khi khách hàng nhập đúng/sai mã CVV hoặc có/không có xác thực 3D Secure?
query_3 = spark.sql("""
    WITH FraudStats AS (
        SELECT
            CASE
                WHEN cvv_result = 1 THEN 'CVV Đúng (Correct)'
                WHEN cvv_result = 0 THEN 'CVV Sai (Incorrect)'
                ELSE 'Không rõ'
            END AS ket_qua_cvv,
            CASE
                WHEN three_ds_flag = 1 THEN 'Có 3DS'
                ELSE 'Không có 3DS'
            END AS xac_thuc_3ds,
            is_fraud
        FROM ecommerce_transactions
    )
    SELECT
        ket_qua_cvv,
        xac_thuc_3ds,
        COUNT(*) AS tong_giao_dich,
        SUM(is_fraud) AS so_giao_dich_gian_lan,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM FraudStats
    GROUP BY ket_qua_cvv, xac_thuc_3ds
    ORDER BY ket_qua_cvv, xac_thuc_3ds;
""")
query_3.show()

+-------------------+------------+--------------+---------------------+--------------+
|        ket_qua_cvv|xac_thuc_3ds|tong_giao_dich|so_giao_dich_gian_lan|ty_le_gian_lan|
+-------------------+------------+--------------+---------------------+--------------+
|CVV Sai (Incorrect)|      Có 3DS|         10758|                  404|          3.76|
|CVV Sai (Incorrect)|Không có 3DS|         27570|                 3661|         13.28|
| CVV Đúng (Correct)|      Có 3DS|        224379|                 1849|          0.82|
| CVV Đúng (Correct)|Không có 3DS|         36988|                  698|          1.89|
+-------------------+------------+--------------+---------------------+--------------+



In [11]:
# 4. Phân tích số lượng giao dịch gian lận theo từng khung giờ trong ngày và theo quốc gia nào có tỷ lệ gian lận cao nhất?
query_4 = spark.sql("""
    SELECT
        HOUR(transaction_time) AS khung_gio_trong_ngay,
        country,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS so_luong_gian_lan,
        COUNT(*) AS tong_giao_dich,
        CAST(SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM ecommerce_transactions
    WHERE country IS NOT NULL AND country != 'Unknown'
    GROUP BY khung_gio_trong_ngay, country
    ORDER BY so_luong_gian_lan DESC
    LIMIT 10;
""")
query_4.show()

+--------------------+-------+-----------------+--------------+--------------+
|khung_gio_trong_ngay|country|so_luong_gian_lan|tong_giao_dich|ty_le_gian_lan|
+--------------------+-------+-----------------+--------------+--------------+
|                  23|     TR|               47|          1213|          3.87|
|                  10|     TR|               43|          1264|          3.40|
|                   8|     US|               42|          1349|          3.11|
|                  11|     TR|               42|          1231|          3.41|
|                  19|     GB|               41|          1320|          3.11|
|                   7|     TR|               41|          1258|          3.26|
|                  17|     PL|               40|          1208|          3.31|
|                  16|     TR|               40|          1197|          3.34|
|                   9|     TR|               40|          1155|          3.46|
|                  11|     PL|               40|    

In [12]:
# 5. Tỷ lệ gian lận ở các tài khoản mới (ví dụ: dưới 30 ngày tuổi) so với các tài khoản cũ hơn như thế nào?
query_5 = spark.sql("""
    SELECT
        CASE
            WHEN account_age_days <= 30 THEN '0-30 ngày (Tài khoản mới)'
            WHEN account_age_days > 30 AND account_age_days <= 180 THEN '31-180 ngày (Tương đối mới)'
            WHEN account_age_days > 180 AND account_age_days <= 365 THEN '181-365 ngày (Tương đối cũ)'
            ELSE '> 365 ngày (Tài khoản cũ)'
        END AS nhom_tuoi_tai_khoan,
        COUNT(*) AS tong_giao_dich,
        SUM(is_fraud) AS so_giao_dich_gian_lan,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM ecommerce_transactions
    GROUP BY nhom_tuoi_tai_khoan
    ORDER BY ty_le_gian_lan DESC;
""")
query_5.show(truncate=False)

+---------------------------+--------------+---------------------+--------------+
|nhom_tuoi_tai_khoan        |tong_giao_dich|so_giao_dich_gian_lan|ty_le_gian_lan|
+---------------------------+--------------+---------------------+--------------+
|0-30 ngày (Tài khoản mới)  |2081          |1051                 |50.50         |
|31-180 ngày (Tương đối mới)|17512         |2205                 |12.59         |
|> 365 ngày (Tài khoản cũ)  |250488        |3008                 |1.20          |
|181-365 ngày (Tương đối cũ)|29614         |348                  |1.18          |
+---------------------------+--------------+---------------------+--------------+



In [13]:
# 6. So sánh tỷ lệ gian lận giữa các giao dịch có giá trị "bình thường" (ví dụ: nhỏ hơn 5 lần mức trung bình của người dùng) và các giao dịch "bất thường" (lớn hơn 5 lần mức trung bình).
query_6 = spark.sql("""
    SELECT
        CASE
            WHEN amount > (avg_amount_user * 5) THEN 'Bất thường (Amount > 5 * Avg)'
            ELSE 'Bình thường'
        END AS loai_chi_tieu,
        COUNT(*) AS tong_giao_dich,
        SUM(is_fraud) AS so_giao_dich_gian_lan,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM ecommerce_transactions
    WHERE  avg_amount_user > 0
    GROUP BY loai_chi_tieu;
""")
query_6.show(truncate=False)

+-----------------------------+--------------+---------------------+--------------+
|loai_chi_tieu                |tong_giao_dich|so_giao_dich_gian_lan|ty_le_gian_lan|
+-----------------------------+--------------+---------------------+--------------+
|Bất thường (Amount > 5 * Avg)|1673          |1503                 |89.84         |
|Bình thường                  |298022        |5109                 |1.71          |
+-----------------------------+--------------+---------------------+--------------+



In [14]:
# 7. Tỷ lệ gian lận khác biệt như thế nào giữa các giao dịch mà quốc gia của khách hàng và quốc gia phát hành thẻ là trùng nhau và không trùng nhau?
query_7 = spark.sql("""
    SELECT
        CASE
            WHEN country = bin_country THEN 'Trùng khớp (Matched)'
            ELSE 'Không trùng khớp (Mismatched)'
        END AS trang_thai_dia_ly,
        COUNT(*) AS tong_giao_dich,
        SUM(is_fraud) AS so_giao_dich_gian_lan,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS ty_le_gian_lan
    FROM ecommerce_transactions
    WHERE
        country IS NOT NULL AND bin_country IS NOT NULL
        AND country != 'Unknown' AND bin_country != 'Unknown'
    GROUP BY trang_thai_dia_ly;
""")
query_7.show(truncate=False)

+-----------------------------+--------------+---------------------+--------------+
|trang_thai_dia_ly            |tong_giao_dich|so_giao_dich_gian_lan|ty_le_gian_lan|
+-----------------------------+--------------+---------------------+--------------+
|Trùng khớp (Matched)         |275965        |3936                 |1.43          |
|Không trùng khớp (Mismatched)|23730         |2676                 |11.28         |
+-----------------------------+--------------+---------------------+--------------+

